# A3 · nb04 — Part 3: scaling on SIFT (100K / 300K / 1M)

**CS4.406 Assignment 3**  ·  prerequisites: `nb00`–`nb03`, and `hypothesis.md` committed

How does the cost of reaching **recall@10 = 0.95** grow with the number of vectors, for HNSW,
IVF-Flat and exact search?

Subsets are the nested, seeded ones nb00 built (100K ⊂ 300K ⊂ 1M, seed 20261023), so a
difference between sizes is attributable to size and not to having sampled a different region
of the space. Ground truth was recomputed per subset on the T4.

### Predictions on record (committed before this ran)

| | prediction | falsified if |
|---|---|---|
| **H3.1** | exact scales as n^-1.0; ~226 qps @100K, ~75 @300K | exponent outside [-1.15, -0.85] |
| **H3.2** | HNSW exponent -0.2 to -0.4; under 4x faster at 100K than at 1M | ratio > 4x or exponent steeper than -0.5 |
| **H3.3** | IVF-Flat strictly between HNSW and exact (-0.5 to -0.8) | not strictly between |
| **H3.4** | HNSW's 3.8x edge over IVF-Flat shrinks below 2.5x at 100K | gap >= 2.5x, or widens as n falls |
| **H3.5** | HNSW build seconds-per-vector rises >= 20% from 100K to 1M | it does not |
| **H3.6** | ANN still beats exact by >10x even at 100K | advantage under 10x |

### Protocol: the search parameter is picked on the TUNE split

This is the part the assignment's query split exists for. For every build setting we walk the
search-parameter grid **on the tune queries (0–4999), recall only**, and stop at the *first*
value reaching 0.95. That value — chosen without ever looking at the eval half — is then timed
on the **eval queries (5000–9999)**, and that timing is the reported number.

Selecting the parameter this way is also much cheaper than timing the whole grid, which is what
makes three sizes x three index types affordable in one session.

Config selection (which `M`, which `nlist`) still compares eval QPS across 3–4 candidates per
index per size. That residual is small: a 5000-query recall mean has a standard error near
0.0014, far below the gaps involved. The report states it.

Builds use all 4 cores; query timing is strictly 1 thread, one query at a time, 3 repeats,
median. Runtime ≈ 1 hour.

## 1 · Setup

In [ ]:
import subprocess, sys

def pip(*a):
    print(subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a],
                         capture_output=True, text=True).stdout[-400:] or "ok")
try:
    import faiss
except ImportError:
    pip("faiss-cpu"); import faiss
for m, p in [("h5py", "h5py"), ("huggingface_hub", "huggingface_hub")]:
    try: __import__(m)
    except ImportError: pip(p)
print("faiss", faiss.__version__)

In [ ]:
%%writefile annbench_core.py
"""
annbench_core.py -- shared core for CS4.406 Assignment 3 (ANN benchmarking).

Every notebook writes an identical copy of this file, so each notebook runs standalone.

Conventions used everywhere in this project:
  * All distances are SQUARED L2, computed on float32 vectors. GloVe is unit-normalised
    first, so squared-L2 order == cosine order. We never rely on the scale of the
    distances stored in the ann-benchmarks HDF5 files (euclidean vs cosine units);
    those files are used for their neighbour IDS only, as an independent cross-check.
  * k = 10 everywhere.
  * Queries 0..4999  = TUNE split (parameter picking).
    Queries 5000..9999 = EVAL split (every number that appears in the report).
  * Timing is single-thread, one query at a time. See bench_single_query.
"""

import json
import os
import platform
import re
import subprocess
import sys
import time
import hashlib

import numpy as np

K = 10
TIE_EPS = 1e-6
TUNE = slice(0, 5000)
EVAL = slice(5000, 10000)


def set_split(n_queries):
    """Re-point the tune/eval halves. The full datasets ship 10,000 queries, but quick mode
    uses fewer, and the halves must move with them so no reported number is ever measured
    on a query that was used for tuning."""
    global TUNE, EVAL
    half = n_queries // 2
    TUNE, EVAL = slice(0, half), slice(half, n_queries)
    return TUNE, EVAL

HF_REPO = "donbosoc/ire-a3-ann"
HF_REPO_TYPE = "dataset"

def _pick_scratch():
    """Scratch for big files: /kaggle/temp does not count against the 20 GB output quota,
    but it is not guaranteed to exist or be writable, so fall back rather than crash.
    A3_SCRATCH overrides it entirely, which is how run_all.sh keeps everything in-tree."""
    for cand in ([os.environ["A3_SCRATCH"]] if os.environ.get("A3_SCRATCH") else []) + [
            "/kaggle/temp/a3", "/kaggle/working/tmp/a3", "/tmp/a3"]:
        try:
            os.makedirs(cand, exist_ok=True)
            probe = os.path.join(cand, ".w")
            with open(probe, "w") as f:
                f.write("x")
            os.remove(probe)
            return cand
        except Exception:
            continue
    raise RuntimeError("no writable scratch directory found")


TMP = _pick_scratch()
OUT = os.environ.get("A3_OUT") or ("/kaggle/working" if os.path.isdir("/kaggle/working")
                                   else os.path.abspath("./out"))


def ensure_dirs():
    for p in (TMP, f"{TMP}/data", f"{TMP}/gt", f"{TMP}/idx",
              f"{OUT}/results", f"{OUT}/figures", f"{OUT}/logs", f"{OUT}/env"):
        os.makedirs(p, exist_ok=True)


# --------------------------------------------------------------------------------------
# Environment
# --------------------------------------------------------------------------------------

def env_fingerprint(verbose=True):
    """Capture the machine + library versions behind every measurement.

    Kaggle VMs are shared and the CPU model can differ between sessions, so this goes
    into every results row and the report states it rather than claiming a quiet machine.
    """
    info = {}
    try:
        cpuinfo = open("/proc/cpuinfo").read()
        models = re.findall(r"^model name\s*:\s*(.+)$", cpuinfo, re.M)
        info["cpu_model"] = models[0].strip() if models else "unknown"
        info["cpu_logical"] = len(models)
    except Exception:
        info["cpu_model"], info["cpu_logical"] = platform.processor() or "unknown", os.cpu_count()
    try:
        mem_kb = int(re.search(r"MemTotal:\s*(\d+)", open("/proc/meminfo").read()).group(1))
        info["mem_gb"] = round(mem_kb / 1048576, 1)
    except Exception:
        info["mem_gb"] = None

    info["python"] = sys.version.split()[0]
    info["numpy"] = np.__version__
    for mod in ("faiss", "torch", "h5py"):
        try:
            info[mod] = __import__(mod).__version__
        except Exception:
            info[mod] = None
    try:
        import torch
        info["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    except Exception:
        info["gpu"] = None
    try:
        info["nvidia_smi"] = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=30).stdout.strip()
    except Exception:
        info["nvidia_smi"] = None

    # Stable id for "which machine produced this row".
    key = f"{info['cpu_model']}|{info['cpu_logical']}|{info['mem_gb']}"
    info["machine_id"] = hashlib.sha1(key.encode()).hexdigest()[:8]

    if verbose:
        print(f"machine_id : {info['machine_id']}")
        print(f"cpu        : {info['cpu_model']}  ({info['cpu_logical']} logical)")
        print(f"memory     : {info['mem_gb']} GB")
        print(f"gpu        : {info['gpu']}")
        print(f"versions   : python {info['python']}, numpy {info['numpy']}, "
              f"faiss {info['faiss']}, torch {info['torch']}, h5py {info['h5py']}")
    return info


def set_single_thread():
    """Pin FAISS to one thread. Every benchmark number in this project is single-thread."""
    import faiss
    faiss.omp_set_num_threads(1)
    n = faiss.omp_get_max_threads()
    assert n == 1, f"expected 1 FAISS thread, got {n}"
    return n


# --------------------------------------------------------------------------------------
# Hugging Face
# --------------------------------------------------------------------------------------

_TOKEN_NAMES = ["HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGINGFACEHUB_API_TOKEN",
                "HF", "hf_token", "huggingface"]


def hf_enabled():
    """HF mirroring is a convenience for running across Kaggle sessions, never a
    requirement. Everything this project writes also lands on local disk, so with no token
    (the normal case for someone reproducing the results) the mirror is simply skipped."""
    if os.environ.get("A3_NO_HF"):
        return False
    try:
        get_hf_token()
        return True
    except Exception:
        return False


def get_hf_token():
    """Find the HF token in Kaggle secrets, then the environment. Raises if absent."""
    tried = []
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for name in _TOKEN_NAMES:
            try:
                tok = client.get_secret(name)
                if tok:
                    print(f"HF token: Kaggle secret '{name}'")
                    return tok.strip()
            except Exception:
                tried.append(f"secret:{name}")
    except Exception:
        tried.append("kaggle_secrets unavailable")
    for name in _TOKEN_NAMES:
        tok = os.environ.get(name)
        if tok:
            print(f"HF token: environment variable '{name}'")
            return tok.strip()
        tried.append(f"env:{name}")
    raise RuntimeError(
        "No Hugging Face token found. Add it under Notebook > Add-ons > Secrets with "
        f"label 'HF_TOKEN' and attach it to this notebook. Tried: {tried}")


def hf_api(token=None):
    from huggingface_hub import HfApi
    return HfApi(token=token or get_hf_token())


def hf_ensure_repo(token=None):
    api = hf_api(token)
    api.create_repo(HF_REPO, repo_type=HF_REPO_TYPE, exist_ok=True, private=False)
    return api


def hf_upload(local_path, path_in_repo, token=None, msg=None, quiet=False, timeout=420):
    """Upload one file, but never let a network stall hang the session.

    huggingface_hub has no hard wall-clock cap on an upload, and a stalled connection will
    block forever. A 12-hour Kaggle session was lost to exactly that. The work runs on a
    daemon thread with a join timeout: if it overruns we warn and carry on, because the
    local copy of everything we upload is always intact on disk.
    """
    import threading

    if not hf_enabled():
        return False
    size = os.path.getsize(local_path) / 1e6
    if not quiet:
        print(f"  -> hf://{HF_REPO}/{path_in_repo}  ({size:.1f} MB)", flush=True)

    box = {}

    def work():
        try:
            hf_api(token).upload_file(
                path_or_fileobj=local_path, path_in_repo=path_in_repo,
                repo_id=HF_REPO, repo_type=HF_REPO_TYPE,
                commit_message=msg or f"add {path_in_repo}")
            box["ok"] = True
        except Exception as exc:
            box["err"] = exc

    th = threading.Thread(target=work, daemon=True)
    th.start()
    # Big files legitimately take longer, so scale the allowance with size.
    th.join(timeout + size * 2)
    if th.is_alive():
        print(f"  ! upload of {path_in_repo} still running after "
              f"{timeout + size * 2:.0f}s -- continuing; the local copy is intact",
              flush=True)
        return False
    if "err" in box:
        print(f"  ! upload of {path_in_repo} failed: {box['err']}", flush=True)
        return False
    return True


def hf_download(path_in_repo, token=None, local_dir=None):
    """Return a local path for a repo file, or None if it is not in the repo."""
    from huggingface_hub import hf_hub_download
    if not hf_enabled():
        return None
    try:
        return hf_hub_download(repo_id=HF_REPO, repo_type=HF_REPO_TYPE,
                               filename=path_in_repo, token=token or get_hf_token(),
                               local_dir=local_dir or f"{TMP}/hf")
    except Exception as e:
        if "404" in str(e) or "EntryNotFound" in type(e).__name__:
            return None
        raise


def fetch(repo_path, token=None, verbose=True):
    """Download one prepared artifact from the HF mirror, erroring clearly if nb00 never ran."""
    p = hf_download(repo_path, token=token, local_dir=f"{TMP}/hf")
    if p is None:
        raise FileNotFoundError(
            f"hf://{HF_REPO}/{repo_path} not found -- run nb00_setup_data first")
    if verbose:
        print(f"  {repo_path:44s} {human(os.path.getsize(p))}")
    return p


def load_local(name, with_base=True, with_stored=False):
    """Load what prepare_data.py wrote to local disk (the path run_all.sh uses)."""
    d = {}
    if with_base:
        d["base"] = np.load(f"{TMP}/data/{name}_base.npy")
    d["query"] = np.load(f"{TMP}/data/{name}_query.npy")
    d["gt_ids"] = np.load(f"{TMP}/gt/{name}_top100_ids.npy")
    d["gt_dists"] = np.load(f"{TMP}/gt/{name}_top100_dists.npy")
    if with_stored:
        d["stored"] = np.load(f"{TMP}/data/{name}_stored_neighbors.npy")
    return d


def load_prepared(name, token=None, with_base=True, with_stored=False, verbose=True):
    """Load a dataset prepared by nb00: base, all 10k queries, and our exact top-100 GT.

    Use d["query"][TUNE] for parameter picking and d["query"][EVAL] for reported numbers.
    """
    d = {}
    if with_base:
        d["base"] = np.load(fetch(f"data/{name}_base.npy", token, verbose))
    d["query"] = np.load(fetch(f"data/{name}_query.npy", token, verbose))
    d["gt_ids"] = np.load(fetch(f"gt/{name}_top100_ids.npy", token, verbose))
    d["gt_dists"] = np.load(fetch(f"gt/{name}_top100_dists.npy", token, verbose))
    if with_stored:
        d["stored"] = np.load(fetch(f"data/{name}_stored_neighbors.npy", token, verbose))
    return d


# --------------------------------------------------------------------------------------
# Exact ground truth on the GPU
# --------------------------------------------------------------------------------------

def exact_gt_gpu(base, queries, k=100, chunk=None, mem_budget_gb=None,
                 refine=True, verbose=True):
    """Brute-force top-k by squared L2, selected on the GPU, then refined exactly.

    The GPU pass uses the (|q|^2 + |b|^2 - 2 q.b) expansion in float32, which is fast but
    loses a few ULPs. Selection errors from that can only happen between points that are
    equal to within those ULPs -- i.e. ties, which our recall metric forgives anyway. The
    returned DISTANCES are then recomputed by direct subtraction in float64, so the
    threshold that recall@10 depends on is exact.

    The distance block is (nq, chunk) float32, which at 10k queries is 400 MB per 10k
    base vectors, so a fixed chunk would OOM a 16 GB T4 outright. `chunk` is derived from
    the free VRAM and the query count unless given explicitly, and the block is written
    into ONE preallocated buffer with in-place ops -- writing the expression out as
    `q_sq + b_sq - 2*q@b.T` would allocate three such blocks per iteration instead.

    Returns (ids int32 (nq,k), dists float64 (nq,k)) sorted ascending.
    """
    import torch
    assert base.dtype == np.float32 and queries.dtype == np.float32
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    n, d = base.shape
    nq = queries.shape[0]
    k = min(k, n)

    # torch.topk allocates a workspace about the size of its input, so the real peak is
    # ~2x the distance block. Budget a fifth of free VRAM to leave headroom for that.
    if mem_budget_gb is None:
        if dev == "cuda":
            torch.cuda.empty_cache()
            mem_budget_gb = max(0.25, 0.20 * torch.cuda.mem_get_info()[0] / (1 << 30))
        else:
            mem_budget_gb = 2.0
    auto = max(1024, int(mem_budget_gb * (1 << 30) / (nq * 4)))
    chunk = min(chunk, auto) if chunk else auto
    chunk = min(chunk, n)

    q = torch.from_numpy(np.ascontiguousarray(queries).copy()).to(dev)
    q_sq = (q * q).sum(1, keepdim=True)
    buf = torch.empty((nq, chunk), device=dev, dtype=torch.float32)

    best_d = best_i = None
    t0 = time.perf_counter()
    if verbose:
        print(f"    gt: {n:,} x {nq:,} in blocks of {chunk:,} "
              f"(~{nq * chunk * 4 / 1e9:.2f} GB/block) on {dev}")
    for s in range(0, n, chunk):
        blk = torch.from_numpy(np.ascontiguousarray(base[s:s + chunk]).copy()).to(dev)
        m = blk.shape[0]
        dist = buf[:, :m]
        torch.mm(q, blk.T, out=dist)
        dist.mul_(-2.0).add_(q_sq).add_((blk * blk).sum(1)[None, :]).clamp_(min=0)
        kk = min(k, m)
        dv, di = torch.topk(dist, kk, dim=1, largest=False)
        di = di + s
        if best_d is None:
            best_d, best_i = dv.clone(), di
        else:
            cd, ci = torch.cat([best_d, dv], 1), torch.cat([best_i, di], 1)
            best_d, pos = torch.topk(cd, k, dim=1, largest=False)
            best_i = torch.gather(ci, 1, pos)
            del cd, ci, pos
        del blk, dv, di
        if verbose:
            print(f"\r    gt: {min(s + chunk, n):,}/{n:,} vectors", end="", flush=True)
    if verbose:
        print(f"\r    gt: {n:,}/{n:,} vectors in {time.perf_counter() - t0:.1f}s on {dev}")

    ids = best_i.cpu().numpy().astype(np.int32)
    approx = best_d.cpu().numpy().astype(np.float64)
    del q, q_sq, best_i, best_d, buf
    if dev == "cuda":
        torch.cuda.empty_cache()
    if not refine:
        return ids, approx

    dists = exact_dists_for_ids(base, queries, ids)
    order = np.argsort(dists, axis=1, kind="stable")
    ids = np.take_along_axis(ids, order, axis=1)
    dists = np.take_along_axis(dists, order, axis=1)
    return ids, dists


def exact_gt(base, queries, k=100, verbose=True):
    """Exact top-k, on the GPU when there is one and through FAISS when there is not.

    The GPU path is far faster for the big ground-truth jobs, but torch is not a hard
    dependency: FAISS's own flat index gives the same ids, and in both cases the returned
    distances are recomputed exactly in float64 so the recall threshold is not approximate.
    """
    try:
        import torch
        if torch.cuda.is_available():
            return exact_gt_gpu(base, queries, k=k, verbose=verbose)
    except ImportError:
        pass

    import faiss
    if verbose:
        print(f"    gt: {len(base):,} x {len(queries):,} via faiss (no GPU)", flush=True)
    ix = faiss.IndexFlatL2(base.shape[1])
    ix.add(base)
    _, ids = ix.search(np.ascontiguousarray(queries), min(k, len(base)))
    ids = ids.astype(np.int32)
    dists = exact_dists_for_ids(base, queries, ids)
    order = np.argsort(dists, axis=1, kind="stable")
    return np.take_along_axis(ids, order, axis=1), np.take_along_axis(dists, order, axis=1)


def exact_dists_for_ids(base, queries, ids, block=512):
    """Exact squared-L2 distance from each query to each of its listed ids, in float64.

    Direct subtraction (no q^2+b^2-2qb expansion), so there is no cancellation error.
    ids == -1 (index returned fewer than k) yields +inf. Chunked over queries to bound
    the (block, k, d) temporary.
    """
    nq, kk = ids.shape
    out = np.empty((nq, kk), dtype=np.float64)
    for s in range(0, nq, block):
        e = min(s + block, nq)
        idx = ids[s:e]
        valid = idx >= 0
        vecs = base[np.where(valid, idx, 0)].astype(np.float64)   # (b, k, d)
        diff = vecs - queries[s:e].astype(np.float64)[:, None, :]
        out[s:e] = np.where(valid, np.einsum("bkd,bkd->bk", diff, diff), np.inf)
    return out


# --------------------------------------------------------------------------------------
# Recall@10, tie-aware  (Part 1 asks for this specifically)
# --------------------------------------------------------------------------------------

def recall_at_k(retrieved_ids, retrieved_true_d, gt_dists, k=K, eps=TIE_EPS):
    """Per-query recall@k that does not punish ties at the k-th neighbour.

    A returned id counts as correct when its EXACT distance is <= the exact distance of
    the true k-th neighbour (plus a tiny slack). If the true 10th and 11th neighbours sit
    at the same distance, returning either one is right, so membership of the stored
    top-10 id list is the wrong test -- distance is the right one.

    Duplicate ids are credited once, so an index that returns the same vector twice
    cannot inflate its score.

    retrieved_ids    : (nq, k') ids from the index; -1 means "fewer than k' results".
    retrieved_true_d : (nq, k') EXACT squared-L2 distances for those ids (inf for -1).
    gt_dists         : (nq, >=k) exact squared-L2 distances of the true neighbours, ascending.
    returns          : (nq,) float64 in [0, 1].
    """
    retrieved_ids = np.asarray(retrieved_ids)
    retrieved_true_d = np.asarray(retrieved_true_d, dtype=np.float64)
    gt_dists = np.asarray(gt_dists, dtype=np.float64)
    if retrieved_ids.ndim == 1:
        retrieved_ids = retrieved_ids[None, :]
        retrieved_true_d = retrieved_true_d[None, :]
        gt_dists = gt_dists[None, :]
    assert retrieved_ids.shape == retrieved_true_d.shape
    assert gt_dists.shape[1] >= k, f"need >= {k} ground-truth distances"

    dk = gt_dists[:, k - 1][:, None]
    thresh = dk * (1.0 + eps) + eps           # relative slack, plus absolute for dk == 0
    hit = (retrieved_ids >= 0) & (retrieved_true_d <= thresh)

    # Count DISTINCT hit ids per row.
    masked = np.where(hit, retrieved_ids, -1)
    masked = np.sort(masked, axis=1)
    dup = np.zeros(masked.shape, dtype=bool)
    dup[:, 1:] = masked[:, 1:] == masked[:, :-1]
    n_hit = ((masked >= 0) & ~dup).sum(1)
    return np.minimum(n_hit, k) / float(k)


def recall_from_ids(retrieved_ids, base, queries, gt_dists, k=K):
    """recall_at_k convenience wrapper that recomputes the exact distances itself.

    Needed for IVF-PQ, whose returned distances are approximations of the real ones and
    must never be fed to the recall metric.
    """
    d = exact_dists_for_ids(base, queries, np.asarray(retrieved_ids))
    return recall_at_k(retrieved_ids, d, gt_dists, k=k)


# --------------------------------------------------------------------------------------
# Timing
# --------------------------------------------------------------------------------------

def bench_single_query(search_fn, queries, repeats=3, warmup=50):
    """Time `search_fn` one query at a time, single thread; report the median run.

    Per-query timers cost ~50-100 ns each against queries of 10^2-10^4 us, i.e. under
    0.1%, so QPS is taken as n / sum(per-query latencies) from the same pass that
    produces the latency percentiles.
    """
    nq = len(queries)
    for i in range(min(warmup, nq)):
        search_fn(queries[i:i + 1])

    runs = []
    for _ in range(repeats):
        lat = np.empty(nq, dtype=np.float64)
        for i in range(nq):
            t0 = time.perf_counter_ns()
            search_fn(queries[i:i + 1])
            lat[i] = time.perf_counter_ns() - t0
        runs.append(lat)

    qps = np.array([nq / (r.sum() / 1e9) for r in runs])
    med = runs[int(np.argsort(qps)[len(qps) // 2])]       # latencies of the median run
    return dict(
        qps_median=float(np.median(qps)),
        qps_runs=[round(float(x), 2) for x in qps],
        qps_spread_pct=round(float(100 * (qps.max() - qps.min()) / np.median(qps)), 2),
        lat_p50_ms=float(np.percentile(med, 50) / 1e6),
        lat_p95_ms=float(np.percentile(med, 95) / 1e6),
        n_queries=int(nq), repeats=int(repeats),
    )


def bench_batch(search_fn, queries, repeats=3, warmup=1):
    """Time one call that passes every query at once (Part 1's batched comparison)."""
    nq = len(queries)
    for _ in range(warmup):
        search_fn(queries)
    secs = []
    for _ in range(repeats):
        t0 = time.perf_counter_ns()
        search_fn(queries)
        secs.append((time.perf_counter_ns() - t0) / 1e9)
    secs = np.array(secs)
    return dict(qps_median=float(nq / np.median(secs)),
                qps_runs=[round(float(nq / s), 2) for s in secs],
                wall_s_median=float(np.median(secs)),
                n_queries=int(nq), repeats=int(repeats))


# --------------------------------------------------------------------------------------
# Results table
# --------------------------------------------------------------------------------------

COLUMNS = [
    "run_id", "timestamp", "part", "dataset", "n_vectors", "dim", "metric",
    "index_kind", "build_params", "search_param_name", "search_param_value",
    "recall_at_10", "recall_std", "qps_median", "qps_runs", "qps_spread_pct",
    "lat_p50_ms", "lat_p95_ms", "build_time_s", "train_time_s", "add_time_s",
    "index_size_bytes", "bytes_per_vector", "n_queries", "repeats", "threads",
    "query_split", "machine_id", "cpu_model", "faiss_version", "scanned_frac",
    "rc_at_10", "notes",
]


class ResultsLog:
    """Append-only results table, mirrored to HF after every few rows.

    One row per benchmark run, as the deliverable requires. Resumability comes from this
    file: `done()` lets a restarted notebook skip measurements already in the table,
    which is what makes a killed Kaggle session cheap.
    """

    KEY = ("part", "dataset", "n_vectors", "index_kind", "build_params",
           "search_param_name", "search_param_value")

    def __init__(self, env, token=None, path=f"{OUT}/results/results.csv",
                 repo_path="results/results.csv", flush_every=5, pull=True):
        import pandas as pd
        self.pd, self.env, self.token = pd, env, token
        self.path, self.repo_path, self.flush_every = path, repo_path, flush_every
        self._since_flush = 0
        os.makedirs(os.path.dirname(path), exist_ok=True)

        df = None
        if pull and hf_enabled():
            try:
                remote = hf_download(repo_path, token=token)
                if remote:
                    df = pd.read_csv(remote)
                    print(f"results.csv: pulled {len(df)} existing rows from HF")
            except Exception as e:
                print(f"results.csv: could not pull from HF ({e}); starting fresh")
        if df is None and os.path.exists(path):
            df = pd.read_csv(path)
            print(f"results.csv: {len(df)} existing rows found locally")
        if df is None:
            df = pd.DataFrame(columns=COLUMNS)
            print("results.csv: starting a new table")
        for c in COLUMNS:
            if c not in df.columns:
                df[c] = None
        self.df = df[COLUMNS]
        self.df.to_csv(self.path, index=False)

    @staticmethod
    def _sv(x):
        """Normalise a search-param value for comparison: pandas may read an all-numeric
        column back as float, turning 40 into '40.0' and breaking resume detection."""
        t = str(x)
        if t.endswith(".0") and t[:-2].lstrip("-").isdigit():
            t = t[:-2]
        return t

    @staticmethod
    def _norm(params):
        return json.dumps(params, sort_keys=True, separators=(",", ":")) \
            if isinstance(params, dict) else str(params)

    def done(self, part, dataset, n_vectors, index_kind, build_params,
             search_param_name, search_param_value):
        """True when this exact measurement is already in the table."""
        if len(self.df) == 0:
            return False
        want = (str(part), str(dataset), self._sv(n_vectors), str(index_kind),
                self._norm(build_params), str(search_param_name),
                self._sv(search_param_value))
        have = self.df[list(self.KEY)].astype(str).apply(
            lambda col: col.map(self._sv) if col.name in ("n_vectors", "search_param_value")
            else col)
        return bool((have == self.pd.Series(want, index=list(self.KEY))).all(axis=1).any())

    def add(self, *, part, dataset, n_vectors, dim, metric, index_kind, build_params,
            search_param_name, search_param_value, recall, timing, build_time_s=None,
            train_time_s=None, add_time_s=None, index_size_bytes=None,
            query_split="eval", threads=1, scanned_frac=None, rc_at_10=None,
            notes="", flush=False):
        recall = np.asarray(recall, dtype=np.float64)
        timing = timing or {}
        row = {
            "run_id": f"{int(time.time() * 1000)}-{np.random.randint(1e6):06d}",
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
            "part": part, "dataset": dataset, "n_vectors": int(n_vectors), "dim": int(dim),
            "metric": metric, "index_kind": index_kind,
            "build_params": self._norm(build_params),
            "search_param_name": search_param_name, "search_param_value": search_param_value,
            "recall_at_10": round(float(recall.mean()), 6),
            "recall_std": round(float(recall.std()), 6),
            "qps_median": (round(float(timing["qps_median"]), 3)
                           if timing.get("qps_median") is not None else None),
            "qps_runs": json.dumps(timing.get("qps_runs")),
            "qps_spread_pct": timing.get("qps_spread_pct"),
            "lat_p50_ms": timing.get("lat_p50_ms"),
            "lat_p95_ms": timing.get("lat_p95_ms"),
            "build_time_s": build_time_s, "train_time_s": train_time_s,
            "add_time_s": add_time_s, "index_size_bytes": index_size_bytes,
            "bytes_per_vector": (round(index_size_bytes / n_vectors, 2)
                                 if index_size_bytes else None),
            "n_queries": timing.get("n_queries"), "repeats": timing.get("repeats"),
            "threads": int(threads), "query_split": query_split,
            "machine_id": self.env.get("machine_id"), "cpu_model": self.env.get("cpu_model"),
            "faiss_version": self.env.get("faiss"), "scanned_frac": scanned_frac,
            "rc_at_10": rc_at_10, "notes": notes,
        }
        # .loc[] append rather than concat: concat against a frame holding all-NA columns
        # raises a FutureWarning on every single row, which floods the notebook log.
        self.df.loc[len(self.df)] = self.pd.Series(row)
        self.df.to_csv(self.path, index=False)
        self._since_flush += 1
        if flush or self._since_flush >= self.flush_every:
            self.flush()
        return row

    def flush(self):
        self._since_flush = 0
        if not hf_enabled():
            return
        t0 = time.time()
        try:
            hf_upload(self.path, self.repo_path, token=self.token,
                      msg=f"results.csv: {len(self.df)} rows", quiet=True)
        except Exception as e:
            print(f"  ! results.csv upload failed ({e}); the local copy is intact")
        dt = time.time() - t0
        if dt > 30:
            print(f"  (results.csv sync took {dt:.0f}s)", flush=True)


# --------------------------------------------------------------------------------------
# Misc
# --------------------------------------------------------------------------------------

def index_size_bytes(index, path=None):
    """Serialise a FAISS index to disk and measure it. Memory footprint ~= file size for
    these index types, and it is the one number we can take reproducibly."""
    import faiss
    path = path or f"{TMP}/idx/_size_probe.faiss"
    faiss.write_index(index, path)
    n = os.path.getsize(path)
    os.remove(path)
    return n


def normalise(x):
    """Unit-normalise rows in place-ish. Zero rows are left as zeros."""
    x = np.ascontiguousarray(x, dtype=np.float32)
    nrm = np.linalg.norm(x, axis=1, keepdims=True)
    nrm[nrm == 0] = 1.0
    return (x / nrm).astype(np.float32)


def relative_contrast(gt_dists, mean_dist_sq, k=1):
    """RC = (mean distance to a random point) / (distance to the k-th neighbour).

    Reported on true (non-squared) distances, the usual convention. Values near 1 mean
    the nearest neighbour is barely nearer than anything else, which is exactly when
    graph and cell indexes stop working.
    """
    dk = np.sqrt(np.asarray(gt_dists, dtype=np.float64)[:, k - 1])
    dk = np.where(dk == 0, np.finfo(np.float64).tiny, dk)
    return np.sqrt(np.asarray(mean_dist_sq, dtype=np.float64)) / dk


def mean_sq_dist_to_random(base, queries, n_sample=100_000, seed=0):
    """Mean squared distance from each query to a random sample of the base."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(base), size=min(n_sample, len(base)), replace=False)
    sample = base[np.sort(idx)].astype(np.float32)
    s_sq = (sample * sample).sum(1)
    out = np.empty(len(queries), dtype=np.float64)
    for s in range(0, len(queries), 512):
        e = min(s + 512, len(queries))
        q = queries[s:e].astype(np.float32)
        d = (q * q).sum(1)[:, None] + s_sq[None, :] - 2.0 * (q @ sample.T)
        out[s:e] = np.maximum(d, 0).mean(1)
    return out


def human(n):
    for unit in ("B", "KB", "MB", "GB"):
        if abs(n) < 1024 or unit == "GB":
            return f"{n:.1f} {unit}"
        n /= 1024

In [ ]:
%%writefile test_recall.py
"""Unit tests for the tie-aware recall@10 metric (Part 1).

Run: pytest -q test_recall.py      (or: python test_recall.py)
"""

import numpy as np
import pytest

from annbench_core import (K, exact_dists_for_ids, recall_at_k, recall_from_ids)


# ---------------------------------------------------------------------------
# A tiny hand-built dataset where we know every distance exactly.
# 1-D points at 0,1,2,...  query at 0  ->  neighbours are 0,1,2,... in order.
# ---------------------------------------------------------------------------

@pytest.fixture
def line():
    base = np.arange(20, dtype=np.float32).reshape(-1, 1)
    queries = np.zeros((1, 1), dtype=np.float32)
    gt_ids = np.arange(K, dtype=np.int32)[None, :]
    gt_dists = exact_dists_for_ids(base, queries, gt_ids)
    return base, queries, gt_ids, gt_dists


def test_exact_result_is_one(line):
    base, queries, gt_ids, gt_dists = line
    assert recall_from_ids(gt_ids, base, queries, gt_dists)[0] == 1.0


def test_permuted_top10_is_one(line):
    """Recall@10 is a set measure: order within the top 10 must not matter."""
    base, queries, gt_ids, gt_dists = line
    perm = gt_ids[:, ::-1].copy()
    assert recall_from_ids(perm, base, queries, gt_dists)[0] == 1.0


def test_one_genuine_miss_is_point_nine(line):
    base, queries, gt_ids, gt_dists = line
    got = gt_ids.copy()
    got[0, 9] = 15                      # true distance 225 >> d10 = 81
    assert recall_from_ids(got, base, queries, gt_dists)[0] == pytest.approx(0.9)


def test_all_wrong_is_zero(line):
    base, queries, _, gt_dists = line
    got = np.arange(10, 20, dtype=np.int32)[None, :]
    assert recall_from_ids(got, base, queries, gt_dists)[0] == 0.0


def test_tie_at_tenth_neighbour_is_not_a_mistake():
    """THE case the assignment singles out.

    Points at +-1..+-5 give five exact distance ties. The true 10th neighbour is at
    distance 25, and so is the 11th: both +5 and -5 sit there. An index that returns -5
    where the stored list says +5 has made no error, and must still score 1.0.
    """
    base = np.array([[0.], [1.], [-1.], [2.], [-2.], [3.], [-3.], [4.], [-4.],
                     [5.], [-5.]], dtype=np.float32)
    queries = np.zeros((1, 1), dtype=np.float32)
    stored = np.arange(K, dtype=np.int32)[None, :]          # ids 0..9, i.e. +5 not -5
    gt_dists = exact_dists_for_ids(base, queries, stored)
    assert gt_dists[0, K - 1] == 25.0

    swapped = stored.copy()
    swapped[0, 9] = 10                                      # return -5 instead of +5
    assert recall_from_ids(swapped, base, queries, gt_dists)[0] == 1.0, \
        "a tie at the 10th neighbour was wrongly counted as a mistake"

    # And a naive id-set metric would have scored this 0.9 -- confirm the distinction
    # is real rather than vacuous.
    naive = len(set(swapped[0].tolist()) & set(stored[0].tolist())) / K
    assert naive == 0.9


def test_all_distances_identical_is_one():
    """Degenerate but legal: every point is equidistant, so any 10 ids are correct."""
    base = np.ones((50, 3), dtype=np.float32)
    queries = np.zeros((1, 3), dtype=np.float32)
    gt_dists = exact_dists_for_ids(base, queries, np.arange(K, dtype=np.int32)[None, :])
    got = np.arange(40, 50, dtype=np.int32)[None, :]
    assert recall_from_ids(got, base, queries, gt_dists)[0] == 1.0


def test_zero_distance_ground_truth():
    """d10 == 0 (duplicated vectors): the relative slack vanishes, so the absolute
    slack has to carry the comparison. Guards against a divide-by-zero style bug."""
    base = np.zeros((20, 4), dtype=np.float32)
    base[15:] = 7.0
    queries = np.zeros((1, 4), dtype=np.float32)
    gt_dists = exact_dists_for_ids(base, queries, np.arange(K, dtype=np.int32)[None, :])
    assert gt_dists[0, K - 1] == 0.0
    assert recall_from_ids(np.arange(5, 15, dtype=np.int32)[None, :],
                           base, queries, gt_dists)[0] == 1.0
    assert recall_from_ids(np.arange(10, 20, dtype=np.int32)[None, :],
                           base, queries, gt_dists)[0] == pytest.approx(0.5)


def test_duplicate_ids_are_credited_once(line):
    """An index returning the same vector ten times must score 0.1, not 1.0."""
    base, queries, _, gt_dists = line
    got = np.zeros((1, K), dtype=np.int32)
    assert recall_from_ids(got, base, queries, gt_dists)[0] == pytest.approx(0.1)


def test_missing_results_are_not_hits(line):
    """FAISS pads with -1 when an index finds fewer than k candidates."""
    base, queries, gt_ids, gt_dists = line
    got = gt_ids.copy()
    got[0, 5:] = -1
    assert recall_from_ids(got, base, queries, gt_dists)[0] == pytest.approx(0.5)


def test_pq_style_approximate_distances_do_not_corrupt_recall(line):
    """IVF-PQ returns approximate distances. recall_from_ids must ignore whatever the
    index reports and recompute from the base vectors."""
    base, queries, gt_ids, gt_dists = line
    lying = np.zeros((1, K), dtype=np.float64)              # index claims every hit is perfect
    wrong_ids = np.arange(10, 20, dtype=np.int32)[None, :]  # ...but they are all wrong
    assert recall_at_k(wrong_ids, lying, gt_dists)[0] == 1.0          # believing the lie
    assert recall_from_ids(wrong_ids, base, queries, gt_dists)[0] == 0.0   # recomputing


def test_matches_bruteforce_on_random_data():
    """Cross-check against an independent, deliberately naive implementation on random
    float data, where exact ties essentially never occur so the two must agree."""
    rng = np.random.default_rng(0)
    base = rng.standard_normal((400, 8), dtype=np.float32)
    queries = rng.standard_normal((25, 8), dtype=np.float32)

    full = ((queries[:, None, :] - base[None, :, :]) ** 2).sum(-1).astype(np.float64)
    order = np.argsort(full, axis=1)
    gt_ids = order[:, :K].astype(np.int32)
    gt_dists = np.take_along_axis(full, order, axis=1)[:, :K]

    got = order[:, rng.permutation(30)[:K]].astype(np.int32)   # a sloppy "index"
    mine = recall_from_ids(got, base, queries, gt_dists)
    naive = np.array([len(set(g.tolist()) & set(t.tolist())) / K
                      for g, t in zip(got, gt_ids)])
    assert np.allclose(mine, naive)


def test_batch_shape_and_mean(line):
    base, queries, gt_ids, gt_dists = line
    q = np.repeat(queries, 4, axis=0)
    gtd = np.repeat(gt_dists, 4, axis=0)
    got = np.repeat(gt_ids, 4, axis=0).copy()
    got[2, 9] = 19
    r = recall_from_ids(got, base, q, gtd)
    assert r.shape == (4,)
    assert r.mean() == pytest.approx((1 + 1 + 0.9 + 1) / 4)


if __name__ == "__main__":
    raise SystemExit(pytest.main([__file__, "-q"]))

In [ ]:
import os, sys, json, time, gc, math
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, os.getcwd())
import annbench_core as C

C.ensure_dirs()
ENV = C.env_fingerprint()
TOKEN = C.get_hf_token()
LOG = C.ResultsLog(ENV, token=TOKEN)
BUILD_THREADS = ENV["cpu_logical"]
print(f"\nmachine_id {ENV['machine_id']}  (nb00-nb03 ran on dc469939)")
if ENV["machine_id"] != "dc469939":
    print("  ! different machine -- Part 3 QPS will not be comparable with Part 2; note it")

## 2 · Data and the nested subsets

Only the ids of each subset were stored, so the vectors come from fancy-indexing the 1M base.
Ground-truth ids for a subset are **local** (0..n-1), matching what an index built on that
subset returns.

In [ ]:
D = C.load_prepared("sift", token=TOKEN, with_base=True)
base_full, query = D["base"], D["query"]
q_tune = np.ascontiguousarray(query[C.TUNE])
q_eval = np.ascontiguousarray(query[C.EVAL])

SIZES = [100_000, 300_000, 1_000_000]
SUB = {}
for n in SIZES:
    if n == len(base_full):
        sub, gt = base_full, D["gt_dists"]
    else:
        tag = f"sift{n // 1000}k"
        ids = np.load(C.fetch(f"data/{tag}_ids.npy", TOKEN))
        sub = np.ascontiguousarray(base_full[ids])
        gt = np.load(C.fetch(f"gt/{tag}_top100_dists.npy", TOKEN))
        assert len(ids) == n
    SUB[n] = dict(base=sub, gt_tune=gt[C.TUNE], gt_eval=gt[C.EVAL])
    print(f"n={n:>9,}  base {sub.shape}  {C.human(sub.nbytes)}")

# nesting sanity: the subsets really are prefixes of one permutation
i100 = set(np.load(C.fetch("data/sift100k_ids.npy", TOKEN, verbose=False)).tolist())
i300 = set(np.load(C.fetch("data/sift300k_ids.npy", TOKEN, verbose=False)).tolist())
assert i100 <= i300, "subsets are not nested"
print("\nsubsets nested: 100K subset of 300K subset of 1M")

## 3 · Exact search at each size

The reference line. Timed exactly as in nb01: one query at a time, one thread, 3 repeats,
median, on a 1000-query sample of the eval half (at ~22 qps the full 5000 would cost 45 min
per size for no extra precision).

In [ ]:
K = 10
REPEATS = 3
N_SINGLE = 1000
exact_rows = {}

for n in SIZES:
    s = SUB[n]
    if LOG.done("3", "sift", n, "exact_flat", {}, "query_mode", "one_at_a_time"):
        prev = LOG.df[(LOG.df.part.astype(str) == "3") & (LOG.df.n_vectors.astype(str) == str(n))
                      & (LOG.df.index_kind == "exact_flat")]
        exact_rows[n] = float(prev.qps_median.iloc[0])
        print(f"n={n:>9,}  exact {exact_rows[n]:8.2f} qps  (from results.csv)")
        continue
    C.set_single_thread()
    ix = faiss.IndexFlatL2(ix_dim := s["base"].shape[1])
    t0 = time.perf_counter(); ix.add(s["base"]); t_add = time.perf_counter() - t0
    size = C.index_size_bytes(ix)
    qs = q_eval[:N_SINGLE]
    _, I = ix.search(qs, K)
    rec = C.recall_from_ids(I, s["base"], qs, s["gt_eval"][:N_SINGLE])
    assert rec.mean() == 1.0, "exact search must score 1.0 against its own ground truth"
    t = C.bench_single_query(lambda x: ix.search(x, K), qs, repeats=REPEATS, warmup=20)
    LOG.add(part="3", dataset="sift", n_vectors=n, dim=ix_dim, metric="l2",
            index_kind="exact_flat", build_params={}, search_param_name="query_mode",
            search_param_value="one_at_a_time", recall=rec, timing=t, build_time_s=t_add,
            add_time_s=t_add, index_size_bytes=size, threads=1,
            notes=f"scaling reference; first {N_SINGLE} eval queries")
    exact_rows[n] = t["qps_median"]
    print(f"n={n:>9,}  exact {t['qps_median']:8.2f} qps · p50 {t['lat_p50_ms']:7.2f} ms · "
          f"{C.human(size)}")
    del ix; gc.collect()
LOG.flush()

## 4 · Harness: screen on tune, time on eval

`cheapest_at_target` walks the grid on the tune queries and returns the first parameter value
whose **tune** recall reaches 0.95, or `None` if the grid never gets there. Only that single
point is then timed on eval. Screening is recall-only and multi-threaded, so it is cheap; the
one timed measurement is strictly single-threaded.

In [ ]:
TARGET = 0.95

def recall_on(index, queries, gt_d, base_arr):
    faiss.omp_set_num_threads(BUILD_THREADS)
    _, I = index.search(queries, K)
    faiss.omp_set_num_threads(1)
    return C.recall_from_ids(I, base_arr, queries, gt_d)

def cheapest_at_target(index, n, grid, setter, label):
    '''First grid value whose TUNE recall reaches TARGET. Returns (value, tune_recall, trace).'''
    s = SUB[n]
    trace = []
    for v in grid:
        setter(index, v)
        r = float(recall_on(index, q_tune, s["gt_tune"], s["base"]).mean())
        trace.append((v, r))
        if r >= TARGET:
            print(f"    tune screen {label}: {len(trace)} points, picked {v} "
                  f"(tune recall {r:.4f})")
            return v, r, trace
    print(f"    tune screen {label}: NEVER reached {TARGET} "
          f"(best {max(t[1] for t in trace):.4f} at {max(trace, key=lambda t: t[1])[0]})")
    return None, max(t[1] for t in trace), trace

def time_point(index, n, kind, bp, pname, v, setter, build_s, size, notes=""):
    s = SUB[n]
    setter(index, v)
    rec = recall_on(index, q_eval, s["gt_eval"], s["base"])
    C.set_single_thread()
    t = C.bench_single_query(lambda x: index.search(x, K), q_eval, repeats=REPEATS, warmup=30)
    LOG.add(part="3", dataset="sift", n_vectors=n, dim=s["base"].shape[1], metric="l2",
            index_kind=kind, build_params=bp, search_param_name=pname, search_param_value=v,
            recall=rec, timing=t, build_time_s=build_s, add_time_s=build_s,
            index_size_bytes=size, threads=1,
            notes=notes or f"param chosen on tune split; build_threads={BUILD_THREADS}")
    print(f"    eval: recall {rec.mean():.4f} · {t['qps_median']:8.1f} qps · "
          f"p50 {t['lat_p50_ms']:.2f} ms · spread {t['qps_spread_pct']}%")
    return dict(n=n, kind=kind, build_params=bp, param=v, recall=float(rec.mean()),
                qps=t["qps_median"], build_s=build_s, size=size)

SCAL = []
print("harness ready")

## 5 · HNSW at each size

`M ∈ {8, 16, 32}` with `efConstruction = 200`, the same set at every size so the comparison is
about n alone. nb02 found `M=16` best on SIFT at 1M; whether that holds at 100K is one of the
things H3.4 is about.

In [ ]:
HNSW_M = [8, 16, 32]
EFC = 200
EF_GRID = [10, 16, 24, 32, 48, 64, 96, 128, 192, 256, 384, 512, 768]

def set_ef(ix, v):
    ix.hnsw.efSearch = int(v)

for n in SIZES:
    s = SUB[n]
    dim = s["base"].shape[1]
    print(f"\n--- HNSW · n={n:,} ---")
    for M in HNSW_M:
        bp = {"M": M, "efConstruction": EFC}
        faiss.omp_set_num_threads(BUILD_THREADS)
        ix = faiss.IndexHNSWFlat(dim, M); ix.hnsw.efConstruction = EFC
        t0 = time.perf_counter(); ix.add(s["base"]); t_build = time.perf_counter() - t0
        faiss.omp_set_num_threads(1)
        size = C.index_size_bytes(ix)
        print(f"  M={M:<3d} build {t_build:7.1f}s ({t_build / n * 1e6:5.1f} us/vector) · "
              f"{C.human(size)} · {size / n:.0f} B/vec")
        v, r, _ = cheapest_at_target(ix, n, EF_GRID, set_ef, f"M={M}")
        if v is not None:
            SCAL.append(time_point(ix, n, "hnsw", bp, "efSearch", v, set_ef, t_build, size))
        else:
            SCAL.append(dict(n=n, kind="hnsw", build_params=bp, param=None, recall=r,
                             qps=None, build_s=t_build, size=size))
        del ix; gc.collect()
    LOG.flush()

## 6 · IVF-Flat at each size

`nlist ∈ {1024, 2048, 4096, 8192}`, again the same set at every size. At 100K, `nlist=8192`
leaves only ~12 vectors per list — a deliberately bad setting that the tune screen is free to
reject, rather than one excluded by hand.

In [ ]:
IVF_NLIST = [1024, 2048, 4096, 8192]
NPROBE_GRID = [1, 2, 4, 8, 12, 16, 24, 32, 48, 64, 96, 128, 192, 256, 384, 512, 768, 1024]

def set_nprobe(ix, v):
    ix.nprobe = int(v)

for n in SIZES:
    s = SUB[n]
    dim = s["base"].shape[1]
    print(f"\n--- IVF-Flat · n={n:,} ---")
    for nlist in IVF_NLIST:
        bp = {"nlist": nlist}
        faiss.omp_set_num_threads(BUILD_THREADS)
        ix = faiss.IndexIVFFlat(faiss.IndexFlatL2(dim), dim, nlist, faiss.METRIC_L2)
        t0 = time.perf_counter(); ix.train(s["base"]); ix.add(s["base"])
        t_build = time.perf_counter() - t0
        faiss.omp_set_num_threads(1)
        size = C.index_size_bytes(ix)
        print(f"  nlist={nlist:<5d} build {t_build:7.1f}s · {C.human(size)} · "
              f"{n // nlist} vectors/list")
        grid = [p for p in NPROBE_GRID if p <= nlist]
        v, r, _ = cheapest_at_target(ix, n, grid, set_nprobe, f"nlist={nlist}")
        if v is not None:
            frac = v / nlist
            note = (f"param chosen on tune split; build_threads={BUILD_THREADS}; "
                    f"nprobe/nlist={frac:.2f}")
            if frac > 0.5:
                print(f"    ! nprobe={v} is {frac:.0%} of nlist -- this IVF has effectively "
                      f"degenerated into a full scan, so its 'cost of 0.95' is really "
                      f"exact-search cost plus quantizer overhead")
                note += " DEGENERATE(>50% of lists probed)"
            SCAL.append(time_point(ix, n, "ivf_flat", bp, "nprobe", v, set_nprobe,
                                   t_build, size, notes=note))
            SCAL[-1]["frac_lists"] = frac
        else:
            SCAL.append(dict(n=n, kind="ivf_flat", build_params=bp, param=None, recall=r,
                             qps=None, build_s=t_build, size=size))
        del ix; gc.collect()
    LOG.flush()

## 7 · Scaling exponents and the verdict on each hypothesis

In [ ]:
import pandas as pd

rows = [r for r in SCAL if r["qps"] is not None]
sc = pd.DataFrame(rows)
best = (sc.sort_values("qps", ascending=False)
          .groupby(["kind", "n"], as_index=False).first())
for n in SIZES:
    best = pd.concat([best, pd.DataFrame([dict(kind="exact_flat", n=n, qps=exact_rows[n],
                                               recall=1.0, param="full scan",
                                               build_params={}, build_s=None, size=None)])],
                     ignore_index=True)

print(f"{'index':11s}{'n':>11s}{'setting':>28s}{'recall':>9s}{'qps':>10s}"
      f"{'build s':>9s}{'B/vec':>8s}")
print("-" * 86)
for kind in ("exact_flat", "ivf_flat", "hnsw"):
    for n in SIZES:
        r = best[(best.kind == kind) & (best.n == n)]
        if not len(r):
            continue
        r = r.iloc[0]
        setting = (f"{json.dumps(r.build_params)} @ {r.param}"
                   if kind != "exact_flat" else "full scan")
        print(f"{kind:11s}{r.n:>11,}{setting:>28s}{r.recall:>9.4f}{r.qps:>10.1f}"
              f"{(f'{r.build_s:.0f}' if r.build_s else '-'):>9s}"
              f"{(f'{r.size / r.n:.0f}' if r.size else '-'):>8s}")

EXP = {}
print(f"\n{'index':11s}{'exponent a in qps ~ n^a':>26s}{'100K/1M ratio':>16s}")
print("-" * 53)
for kind in ("exact_flat", "ivf_flat", "hnsw"):
    g = best[best.kind == kind].sort_values("n")
    if len(g) < 2:
        continue
    a = float(np.polyfit(np.log(g.n.values.astype(float)), np.log(g.qps.values.astype(float)), 1)[0])
    ratio = float(g.qps.iloc[0] / g.qps.iloc[-1])
    EXP[kind] = dict(exponent=a, ratio=ratio)
    print(f"{kind:11s}{a:>26.3f}{ratio:>15.2f}x")

In [ ]:
def verdict(ok):
    return "SUPPORTED" if ok else "NOT SUPPORTED"

print("=" * 86); print("HYPOTHESIS VERDICTS (Part 3)"); print("=" * 86)

e = EXP["exact_flat"]["exponent"]
h31 = -1.15 <= e <= -0.85
print(f"H3.1  exact scales as n^-1.0              exponent {e:+.3f}   -> {verdict(h31)}")
print(f"      H3.1 also predicted (for SIFT) ~226 qps at 100K and ~75 at 300K; "
      f"measured {exact_rows[SIZES[0]]:.1f} and {exact_rows[SIZES[1]]:.1f}")

hx, hr = EXP["hnsw"]["exponent"], EXP["hnsw"]["ratio"]
h32 = hr < 4.0 and hx >= -0.5
print(f"\nH3.2  HNSW exponent -0.2..-0.4, <4x       exponent {hx:+.3f}, ratio {hr:.2f}x"
      f"  -> {verdict(h32)}")

ix_ = EXP["ivf_flat"]["exponent"]
h33 = EXP["hnsw"]["exponent"] > ix_ > e
print(f"\nH3.3  IVF-Flat strictly between           {e:+.3f} < {ix_:+.3f} < {hx:+.3f}"
      f"  -> {verdict(h33)}")

gaps = {}
for n in SIZES:
    h = best[(best.kind == "hnsw") & (best.n == n)]
    i = best[(best.kind == "ivf_flat") & (best.n == n)]
    if len(h) and len(i):
        gaps[n] = float(h.qps.iloc[0]) / float(i.qps.iloc[0])
h34 = gaps.get(100_000, 99) < 2.5 and gaps.get(100_000, 0) < gaps.get(1_000_000, 0)
print(f"\nH3.4  HNSW/IVF gap shrinks below 2.5x@100K  "
      + " ".join(f"{n // 1000}K:{g:.2f}x" for n, g in sorted(gaps.items()))
      + f"  -> {verdict(h34)}")

bpv = {}
for n in SIZES:
    h = best[(best.kind == "hnsw") & (best.n == n)]
    if len(h) and h.build_s.iloc[0]:
        bpv[n] = float(h.build_s.iloc[0]) / n * 1e6
h35 = (bpv.get(1_000_000, 0) / bpv.get(100_000, 1)) >= 1.2 if len(bpv) >= 2 else False
print(f"\nH3.5  HNSW build us/vector rises >=20%      "
      + " ".join(f"{n // 1000}K:{v:.1f}us" for n, v in sorted(bpv.items()))
      + f"  ({bpv.get(1_000_000, 0) / max(bpv.get(100_000, 1), 1e-9):.2f}x)"
      + f"  -> {verdict(h35)}")

adv = {n: float(best[(best.kind == 'hnsw') & (best.n == n)].qps.iloc[0]) / exact_rows[n]
       for n in SIZES if len(best[(best.kind == 'hnsw') & (best.n == n)])}
h36 = min(adv.values()) > 10
print(f"\nH3.6  ANN >10x exact at every size         "
      + " ".join(f"{n // 1000}K:{a:.0f}x" for n, a in sorted(adv.items()))
      + f"  -> {verdict(h36)}")
print("=" * 86)

VERDICTS = dict(H3_1=h31, H3_2=h32, H3_3=h33, H3_4=h34, H3_5=h35, H3_6=h36)

## 8 · Figure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))
style = {"exact_flat": ("crimson", "d--", "exact (full scan)"),
         "ivf_flat": ("#059669", "s-", "IVF-Flat"),
         "hnsw": ("#2563eb", "o-", "HNSW")}

ax = axes[0]
for kind, (col, mk, lab) in style.items():
    g = best[best.kind == kind].sort_values("n")
    if not len(g):
        continue
    ax.plot(g.n, g.qps, mk, color=col, lw=1.8, ms=6,
            label=f"{lab}  (n^{EXP[kind]['exponent']:+.2f})")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("number of vectors"); ax.set_ylabel("queries/s at recall@10 = 0.95")
ax.set_title("Cost of reaching recall 0.95 vs dataset size\n(SIFT, 1 thread, one query at a time)",
             fontsize=10)
ax.grid(alpha=0.25, which="both"); ax.legend(fontsize=8)
ax.set_xticks(SIZES); ax.set_xticklabels([f"{n // 1000}K" if n < 1e6 else "1M" for n in SIZES])

ax = axes[1]
for kind, (col, mk, lab) in style.items():
    g = best[(best.kind == kind) & best.build_s.notna()].sort_values("n")
    if not len(g):
        continue
    ax.plot(g.n, g.build_s.astype(float), mk, color=col, lw=1.8, ms=6, label=lab)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("number of vectors"); ax.set_ylabel("build time (s, 4 threads)")
ax.set_title("Build cost vs dataset size", fontsize=10)
ax.grid(alpha=0.25, which="both"); ax.legend(fontsize=8)
ax.set_xticks(SIZES); ax.set_xticklabels([f"{n // 1000}K" if n < 1e6 else "1M" for n in SIZES])

fig.tight_layout()
fig.savefig(f"{C.OUT}/figures/part3_scaling.png", dpi=150)
plt.show()
C.hf_upload(f"{C.OUT}/figures/part3_scaling.png", "figures/part3_scaling.png",
            token=TOKEN, msg="nb04 part3 figure")

## 9 · Summary — paste this back

In [ ]:
print("=" * 86); print("nb04 SUMMARY (Part 3 · scaling on SIFT)"); print("=" * 86)
print(f"machine_id {ENV['machine_id']} · faiss {ENV['faiss']} · "
      f"results.csv now holds {len(LOG.df)} rows\n")
print(f"{'index':11s}{'100K':>22s}{'300K':>22s}{'1M':>22s}")
print("-" * 79)
for kind in ("exact_flat", "ivf_flat", "hnsw"):
    cells_ = []
    for n in SIZES:
        r = best[(best.kind == kind) & (best.n == n)]
        if not len(r):
            cells_.append("-"); continue
        r = r.iloc[0]
        p = "" if kind == "exact_flat" else f" @{r.param}"
        cells_.append(f"{r.qps:.1f} qps{p}")
    print(f"{kind:11s}" + "".join(c.rjust(22) for c in cells_))

print(f"\n{'index':11s}{'exponent':>12s}{'100K/1M':>10s}")
for kind, v in EXP.items():
    print(f"{kind:11s}{v['exponent']:>+12.3f}{v['ratio']:>9.2f}x")

print("\nbest setting per size")
for kind in ("ivf_flat", "hnsw"):
    for n in SIZES:
        r = best[(best.kind == kind) & (best.n == n)]
        if len(r):
            print(f"  {kind:9s} n={n:>9,}  {json.dumps(r.iloc[0].build_params)} "
                  f"@ {r.iloc[0].param}  (recall {r.iloc[0].recall:.4f})")

deg = [r for r in SCAL if r.get("frac_lists", 0) > 0.5]
if deg:
    print("\nIVF settings that degenerated into near-full scans (nprobe > 50% of nlist)")
    for r in deg:
        print(f"  n={r['n']:>9,} {json.dumps(r['build_params'])} @ nprobe={r['param']} "
              f"({r['frac_lists']:.0%} of lists)")

print("\nconfigs that never reached 0.95 on the tune split")
none_ = [r for r in SCAL if r["qps"] is None]
print("  none" if not none_ else
      "\n".join(f"  {r['kind']:9s} n={r['n']:>9,} {json.dumps(r['build_params'])} "
                f"peak tune recall {r['recall']:.4f}" for r in none_))

print("\nhypothesis verdicts: " + ", ".join(
    f"{k.replace('_', '.')}={'Y' if v else 'N'}" for k, v in VERDICTS.items()))
print("=" * 86)

### Next

`nb05` runs Part 4: the synthetic distance-concentration experiment from `gen_synthetic.py`,
whose hypotheses are already committed. Paste this summary back first — H3.4's measured gaps
tell me how much headroom the Part 4 budgets need, and any `never reached 0.95` rows change
which settings are worth testing there.